# Image Classification with Deep Learning Using TensorFlow

In [ ]:
# Test validates moving from local to cloud / 80% accuracy..

# Google Colab - Cloud Phase

In [ ]:
# dependencies first
!pip install mlflow dagshub tensorflow pandas scikit-learn

In [ ]:
from google.colab import drive
from google.colab import userdata
import zipfile
import os

In [ ]:
# mount drive
drive.mount('/content/drive')

# root of Google Drive
zip_path = '/content/drive/MyDrive/TRAFFIC_SIGN.zip'
extract_path = '/content/data'

if not os.path.exists(extract_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
print("Data ready..")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np
import dagshub
import mlflow
import mlflow.tensorflow

In [ ]:
# secret value
api_key = userdata.get('dagshub_token')

# MLflow Tracking set up
os.environ['MLFLOW_TRACKING_USERNAME'] = "removed user name"
os.environ['MLFLOW_TRACKING_PASSWORD'] = api_key

# initialize connection
dagshub.init(repo_owner='serranoe.e20', repo_name='Traffic-Sign-Project', mlflow=True)
mlflow.tensorflow.autolog()

In [ ]:
# load data
train_df = pd.read_csv('/content/data/Train.csv')
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df['ClassId'] = train_df['ClassId'].astype(str)

# 20% validation split to track real performance vs overfitting
datagen =ImageDataGenerator(rescale=1./255,validation_split=0.2)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/data',
    x_col='Path',
    y_col='ClassId',
    target_size=(32, 32),
    batch_size=64, # 64 batch size
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/data',
    x_col='Path',
    y_col='ClassId',
    target_size=(32, 32),
    batch_size=64,
    class_mode='categorical',
    subset='validation'

In [ ]:
# calculate class weights
print("Calculating class weights to handle imbalance..")

# numerical labels from  generator
true_classes =train_generator.classes
class_indices= train_generator.class_indices

# unique classes
unique_classes =np.unique(true_classes)

# compute  weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=true_classes
)
# package into dict for tf
class_weights_dict = dict(zip(unique_classes, weights))
print("Class weights calculated successfully..")

# Architecture & Training

In [ ]:
# build model
model = models.Sequential([
    layers.Input(shape=(32,32,3)),
    layers.Conv2D(32,(3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # dropout prevent overfitting full dataset
    layers.Dense(43, activation='softmax')
])

# compile & train
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-3,
    first_decay_steps=1000,
    t_mul=2.0,
    m_mul=0.9
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# execute run / opens  notebook & start recording
with mlflow.start_run(run_name="GTSRB_Weighted_Run_02_Shuffled"):
    print("Starting full GPU training phase..")
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=10,
        class_weight=class_weights_dict, # weights applied here
        verbose=1
    )

# Model Testing

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report

In [ ]:
# load test csv
print("Loading test data..")
test_df = pd.read_csv('/content/data/Test.csv')
test_df['ClassId'] = test_df['ClassId'].astype(str)
print("Test data loaded successfully..")

In [ ]:
# test generator / shuffle=False so predicts align with csv rows
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/content/data',
    x_col='Path',
    y_col='ClassId',
    target_size=(32, 32),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# evaluate model
print("\nRunning evaluation on unseen data..")
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"FINAL TEST ACCURACY: {test_accuracy * 100:.2f}%")
print(f"FINAL TEST LOSS:     {test_loss:.4f}")

In [ ]:
# classification report / predict every image in the test set
predictions = model.predict(test_generator, verbose=1)

# convert probabilities into final class choices
predicted_classes = np.argmax(predictions, axis=1)

# true answers
true_classes = test_generator.classes

# breakdown for all 43 classes
print(classification_report(true_classes, predicted_classes))